In [0]:
CREATE OR REPLACE TABLE real_time_catalog.default.product_price_history(
    product_id INT,
    price DECIMAL(10,2),
    updated_at DATE
);

INSERT INTO real_time_catalog.default.product_price_history VALUES
(1, 100.00, '2022-01-01 00:00:00'),
(1, 105.00, '2022-01-02 00:00:00'),
(2, 200.00, '2022-01-01 00:00:00'),
(2, 210.00, '2022-01-02 00:00:00'),
(3, 300.00, '2022-01-01 00:00:00');

Select * from real_time_catalog.default.product_price_history;

# Need to have eff_start and eff_end of product prices the latest price for any product should have eff_end as NULL

In [0]:
Select product_id,price,updated_at as eff_start,lag(updated_at) OVER(Partition by product_id order by updated_at desc) as eff_end 
from real_time_catalog.default.product_price_history;

In [0]:
Select * from real_time_catalog.default.product_price_history;

# Dedup records based on most recent "updated_at"

In [0]:
%python
from pyspark.sql.functions import *
from pyspark.sql.window import *
from pyspark.sql.types import *



df = spark.read.table('real_time_catalog.default.product_price_history')

df = df.withColumn('dedup', dense_rank().over(Window.partitionBy(col("product_id")).orderBy(col('updated_at').desc()))).filter(col("dedup")==1).drop(col("dedup"))

display(df)

### There is an Orders table having columns as "order_id", "order_amount" and "cust_id".
### There is a Refunds table having columns as "refund_id", "order_id" and "refund_amount".
### Orders and Refunds tables have "order_id" as the common key.
### In Orders table one customer can have multiple orders and in refunds table one order can have multiple refunds.
### Please find out the revenue per customer.
### -Revenue is (Order_amount - Refunds Amount)

In [0]:
CREATE OR REPLACE TABLE real_time_catalog.default.orders (
    order_id INT,
    order_amount DECIMAL(10,2),
    cust_id INT
);

CREATE OR REPLACE TABLE real_time_catalog.default.refunds (
    refund_id INT,
    order_id INT,
    refund_amt DECIMAL(10,2)
);

INSERT INTO real_time_catalog.default.orders VALUES
(101, 500.00, 1),
(102, 300.00, 1),
(103, 700.00, 2),
(104, 200.00, 2),
(105, 900.00, 3),
(106, 150.00, 1),
(107, 450.00, 3),
(108, 600.00, 2);

INSERT INTO real_time_catalog.default.refunds VALUES
(1001, 101, 50.00),
(1002, 101, 100.00),
(1003, 102, 30.00),
(1004, 103, 175.00),
(1005, 104, 200.00),
(1006, 105, 90.00),
(1007, 106, 15.00),
(1008, 107, 225.00),
(1009, 108, 60.00),
(1010, 108, 120.00);


SELECT * FROM real_time_catalog.default.orders;

SELECT * FROM real_time_catalog.default.refunds;



In [0]:
SELECT * FROM real_time_catalog.default.orders;

In [0]:
--Solution

WITH CTE1 as
(Select sum(order_amount) as total_amt,order_id,cust_id 
from real_time_catalog.default.orders
group by all order by 1 desc),
CTE2 as
(Select sum(refund_amt) as total_refund,order_id from real_time_catalog.default.refunds group by all order by 1 desc)
Select CTE1.cust_id,sum((total_amt-total_refund)) as rev from CTE1 LEFT join CTE2 ON CTE1.order_id =CTE2.order_id
group by all